In [24]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

RANDOM_STATE = 42
DATA_PATH = "../../data/combined_data.csv"
OUT_DIR = "../../outputs"


In [44]:
df = pd.read_csv('../../data/combined_data.csv')
df.head()


,label,text
0,1,ounce feather bowl hummingbird opec moment ala...
1,1,wulvob get your medircations online qnb ikud v...
2,0,computer connection from cnn com wednesday es...
3,1,university degree obtain a prosperous future m...
4,0,thanks for all your answers guys i know i shou...


In [50]:
import sweetviz as sv
report = sv.analyze(df)
report.show_html("report.html")

Feature: label                               |███▎      | [ 33%]   00:03 -> (00:06 left)2026-08-29 18:25:01,777 - WARNING  - findfont: Failed to find font weight normal, now using 500.
2026-08-29 18:25:01,969 - WARNING  - findfont: Failed to find font weight normal, now using 500.
2026-08-29 18:25:02,243 - WARNING  - findfont: Failed to find font weight normal, now using 500.
2026-08-29 18:25:02,304 - WARNING  - findfont: Failed to find font weight normal, now using 500.
Feature: text                                |██████████| [100%]   00:11 -> (00:00 left)2026-08-29 18:25:09,189 - WARNING  - findfont: Failed to find font weight normal, now using 500.
2026-08-29 18:25:09,221 - WARNING  - findfont: Failed to find font weight normal, now using 500.
[Step 3/3] Generating associations graph     |          | [  0%]   00:00 -> (? left)    2026-08-29 18:25:09,335 - WARNING  - findfont: Failed to find font weight normal, now using 500.
2026-08-29 18:25:09,925 - WARNING  - findfont: Failed to 

Report report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [51]:
import dtale
d = dtale.show(df) 
d


2026-08-31 20:30:30,266 - INFO     - Executing shutdown due to inactivity...
2026-08-31 20:30:42,596 - INFO     - Executing shutdown...
2026-08-31 20:30:42,601 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer


In [25]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")



Loaded shape: (83448, 2)
Label distribution:
label
1    43910
0    39538
Name: count, dtype: int64


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URLTOKEN ", text)   # normalize URLs
    text = re.sub(r"[^a-z\s]", " ", text)                   # strip punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Cleaning text...")
df["clean_text"] = df["text"].apply(clean_text)

X = df["clean_text"]
y = df["label"]




Cleaning text...


In [34]:
cleaned_path = os.path.join(OUT_DIR, "cleaned_combined_data.csv")
df.to_csv(cleaned_path, index=False)
print(f"Cleaned data saved to: {cleaned_path}")

Cleaned data saved to: ../../outputs\cleaned_combined_data.csv


In [35]:
cleaned_path = os.path.join(OUT_DIR, "cleaned_combined_data.csv")
df[["clean_text", "label"]].to_csv(cleaned_path, index=False)
print(f"Cleaned data saved to: {cleaned_path}")

Cleaned data saved to: ../../outputs\cleaned_combined_data.csv


In [27]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train_text.shape[0]}, Test: {X_test_text.shape[0]}")



Train: 66758, Test: 16690


In [52]:
train_df = pd.DataFrame({"clean_text": X_train_text, "label": y_train})
test_df = pd.DataFrame({"clean_text": X_test_text, "label": y_test})

train_path = os.path.join(OUT_DIR, "train_data.csv")
test_path = os.path.join(OUT_DIR, "test_data.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Train data saved to: {train_path}")
print(f"Test data saved to: {test_path}")

Train data saved to: ../../outputs\train_data.csv
Test data saved to: ../../outputs\test_data.csv


In [53]:
print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Total: {len(train_df) + len(test_df)}")
print(f"Train %: {len(train_df) / (len(train_df) + len(test_df)) * 100:.1f}%")
print(f"Test %: {len(test_df) / (len(train_df) + len(test_df)) * 100:.1f}%")

Train rows: 66758
Test rows: 16690
Total: 83448
Train %: 80.0%
Test %: 20.0%


In [28]:
vectorizer = TfidfVectorizer(
    max_features=20000, ngram_range=(1, 2),
    stop_words="english", min_df=3, max_df=0.9
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print(f"TF-IDF matrix: {X_train.shape}")


TF-IDF matrix: (66758, 20000)


In [48]:
import pandas as pd

tfidf_df = pd.DataFrame(
    X_train.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print(f"TF-IDF matrix shape: {tfidf_df.shape}")
tfidf_df.head()

TF-IDF matrix shape: (66758, 20000)


,aa,aaa,aac,aac escapenumber,ab,ab escapenumber,abandon,abandoned,abartlet,abartlet date,...,zugaenglichen,zugaenglichen informationen,zukunft,zum,zum kauf,zur,zurich,zwei,zyban,zz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
sample_df = pd.DataFrame(
    X_train[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)
sample_df

,aa,aaa,aac,aac escapenumber,ab,ab escapenumber,abandon,abandoned,abartlet,abartlet date,...,zugaenglichen,zugaenglichen informationen,zukunft,zum,zum kauf,zur,zurich,zwei,zyban,zz
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
print("=" * 70)
print("MODEL: Logistic Regression")
print("=" * 70)

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

start_time = time.time()
log_reg.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds")

y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]


MODEL: Logistic Regression
Training completed in 1.21 seconds


In [30]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")


Accuracy:  0.9850
Precision: 0.9835
Recall:    0.9882
F1 Score:  0.9858
ROC-AUC:   0.9983


In [31]:
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(cm)
print(f"\n(Rows = actual, Columns = predicted)")
print(f"True Negatives:  {cm[0][0]}  | False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}  | True Positives:  {cm[1][1]}")


Confusion Matrix:
[[7762  146]
 [ 104 8678]]

(Rows = actual, Columns = predicted)
True Negatives:  7762  | False Positives: 146
False Negatives: 104  | True Positives:  8678


In [32]:
import joblib
import os

os.makedirs(OUT_DIR, exist_ok=True)

model_path = os.path.join(OUT_DIR, "phishing_model.pkl")
joblib.dump(log_reg, model_path)
print(f"Model saved to: {model_path}")

vectorizer_path = os.path.join(OUT_DIR, "vectorizer.pkl")
joblib.dump(vectorizer, vectorizer_path)
print(f"Vectorizer saved to: {vectorizer_path}")

Model saved to: ../../outputs\phishing_model.pkl
Vectorizer saved to: ../../outputs\vectorizer.pkl


In [33]:
print(df[df['label'] == 1]['text'].iloc[0][:300])
print("---")
print(df[df['label'] == 0]['text'].iloc[0][:300])

ounce feather bowl hummingbird opec moment alabaster valkyrie dyad bread flack desperate iambic hadron heft quell yoghurt bunkmate divert afterimage
---
 computer connection from cnn com wednesday escapenumber may escapenumber escapenumber escapenumber escapenumber pm edt in this report next generation toys read brain waves google expands personalization youtube wins webby old fashioned rabbit ears laptop review next generation toys read brain waves
